# Análisis de Datos Operativos y Detección de Anomalías en Equipo Industrial
Este notebook desarrolla una solución completa para analizar los datos de operación de un equipo industrial, detectando posibles fallas o comportamientos inusuales utilizando Python y Machine Learning.

Subimos el archivo `MBC_Datos_Sinteticos_Prueba.csv` al entorno antes de ejecutar las celdas.

## 1. Lectura y preparación de datos

En esta etapa, importamos los datos, revisamos su estructura y corregimos los problemas de calidad (valores nulos o duplicados).

In [2]:
import pandas as pd
import numpy as np

df=pd.read_csv('MBC_Datos_Sinteticos_Prueba.csv')

print("-- revision inicial--")
print("Dimensiones originales:", df.shape)
print("Valores nulos de la columna:\n", df.isnull().sum())
print("Valores nulos por columna:\n", df.isnull().sum())
print("Registros duplicados:", df.duplicated().sum())

#Limpieza de data
df_clean=df.drop_duplicates()
df_clean=df_clean.dropna()

#Convertiremos la columna de fecha a formato datetime
df_clean['fecha_hora']=pd.to_datetime(df_clean['fecha_hora'])

print("\n---resultado de limpieza---")
print("Dimensiones finales:", df_clean.shape)
display(df_clean.head())

-- revision inicial--
Dimensiones originales: (240, 5)
Valores nulos de la columna:
 fecha_hora            0
equipo_id             0
rpm                   2
vibracion_rms_mm_s    3
temperatura_c         0
dtype: int64
Valores nulos por columna:
 fecha_hora            0
equipo_id             0
rpm                   2
vibracion_rms_mm_s    3
temperatura_c         0
dtype: int64
Registros duplicados: 0

---resultado de limpieza---
Dimensiones finales: (235, 5)


,fecha_hora,equipo_id,rpm,vibracion_rms_mm_s,temperatura_c
0,2026-05-01 00:00:00,RED-001,436.0,1.884,49.3
1,2026-05-01 06:00:00,RED-001,631.0,2.097,53.0
2,2026-05-01 12:00:00,RED-001,388.0,1.744,48.7
3,2026-05-01 18:00:00,RED-001,388.0,1.839,51.1
4,2026-05-02 00:00:00,RED-001,1357.0,3.406,59.8


**Explicación de la limpieza:**
Se detectaron valores faltantes que representan apenas el ~2% de la muestra. Se optó por eliminar estas filas (`dropna`) en lugar de imputar valores (ej. con la media), ya que en datos de equipos industriales, imputar valores sintéticos puede ocultar comportamientos inusuales reales o confundir al algoritmo de detección de anomalías.

## 2. Análisis de datos
Calcularemos las estadísticas descriptivas para entender cómo opera la máquina en condiciones estándar.

In [3]:
estadisticas=df_clean[['rpm', 'vibracion_rms_mm_s', 'temperatura_c']].describe()
display(estadisticas)

,rpm,vibracion_rms_mm_s,temperatura_c
count,235.000000,235.000000,235.000000
mean,848.970213,2.480519,54.252766
std,325.586905,0.640290,4.710411
min,385.000000,1.447000,44.300000
25%,583.500000,1.972500,50.650000
50%,839.000000,2.401000,53.400000
75%,1121.500000,2.886500,58.100000
max,1382.000000,4.674000,65.900000


**Observaciones del análisis:**
* **Tendencias:** El equipo no trabaja a velocidad constante. La amplia desviación estándar y los percentiles de RPM (desde ~385 hasta ~1380) indican que opera realizando transiciones entre distintos regímenes o "clústeres" de velocidad.
* **Comportamiento:** Existe una correlación directa esperada: a mayores RPM, la base normal de vibración y la temperatura general tienden a incrementar.

## 3. Detaccion de anomalias
Se implementa **Isolation Forest (Bosque de Aislamineto)**, este es perteneciente a un algoritmo del Machine Learning no supervisado ideal para evaluar multiples variables de forma simultanea.

In [4]:
from sklearn.ensemble import IsolationForest

features=['rpm', 'vibracion_rms_mm_s', 'temperatura_c']

#Iniciamos asumiendo el modelo con un 5% de anomalias de los datos
modelo=IsolationForest(contamination=0.05, random_state=42)

#Ajuste y predicciones (-1 es anomalia y 1 es normal)
df_clean['prediccion']=modelo.fit_predict(df_clean[features])
df_clean['es_anomalo']=df_clean['prediccion']==-1

anomalias=df_clean[df_clean['es_anomalo']]
print(f"Total de registros anomalos detectados: {len(anomalias)}\n")
display(anomalias[['fecha_hora', 'rpm', 'vibracion_rms_mm_s', 'temperatura_c']])

Total de registros anomalos detectados: 12



,fecha_hora,rpm,vibracion_rms_mm_s,temperatura_c
36,2026-05-10 00:00:00,592.0,3.598,51.6
76,2026-05-20 00:00:00,1337.0,3.353,65.8
97,2026-05-25 06:00:00,401.0,1.777,44.3
148,2026-06-07 00:00:00,1346.0,3.881,57.8
154,2026-06-08 12:00:00,426.0,3.379,50.1
156,2026-06-09 00:00:00,1328.0,4.674,60.4
160,2026-06-10 00:00:00,425.0,2.966,46.2
164,2026-06-11 00:00:00,412.0,3.105,46.9
170,2026-06-12 12:00:00,1071.0,4.468,56.0
191,2026-06-17 18:00:00,1363.0,3.194,65.9


**Nota:**
El número 42 no tiene ningún impacto matemático especial en el algoritmo; podrías usar el 0, el 1 o el 999 y funcionaría igual. Se utiliza con un propósito fundamental: la reproducibilidad.

Como Isolation Forest es un algoritmo basado en procesos aleatorios (construye árboles haciendo particiones de datos al azar), si no defines un random_state, los resultados variarán ligeramente cada vez que corras la celda. Al fijar una "semilla" (seed) con un número fijo como el 42, obligas al algoritmo a usar la misma secuencia de aleatoriedad, garantizando que tú y cualquier persona que revise tu código obtengan exactamente los mismos registros anómalos en cada ejecución.

**Justificación y limitaciones:**
* **Por qué se eligió:** En maquinaria industrial, un umbral fijo no sirve (por ejemplo: 3.5 mm/s puede ser normal a 1300 RPM, pero indica falla a 400 RPM). Isolation Forest cruza RPM, vibración y temperatura simultáneamente para encontrar estas "incongruencias contextuales".
* **Limitaciones:** Al ser un método no supervisado basado en estadística, aísla puntos atípicos pero no identifica qué pieza exacta está fallando. Requiere validación de un especialista en mantenimiento.

## 4. Visualización de resultados
Utilizaremos `Plotly` para crear gráficos interactivos. Puedes pasar el cursor sobre los puntos para ver el detalle exacto de las anomalías marcadas en rojo.

In [5]:
import plotly.express as px
import plotly.graph_objects as go

fig_evolucion=px.line(df_clean, x='fecha_hora', y='vibracion_rms_mm_s',
                      title='Evolucion de la Vibracion RMS en el tiempo')

fig_evolucion.add_trace(
    go.Scatter(x=anomalias['fecha_hora'], y=anomalias['vibracion_rms_mm_s'],
               mode='markers', marker=dict(color='red', size=10, symbol='x'),
               name='Anomalia', text=anomalias['rpm'],
               hovertemplate='Fecha: %{x}<br>Vibracion: %{y} mm/s<br>RPM: %{text}')
)
fig_evolucion.show()

In [7]:
fig_relacion=px.scatter(df_clean, x='rpm', y='vibracion_rms_mm_s',
                        color='es_anomalo',
                        color_discrete_map={False:'royalblue', True:'red'},
                        title='Relacion entre Velocidad (RPM) y Vibracion RMS',
                        labels={'es_anomalo': '¿Es anomalia?'},
                        hover_data=['fecha_hora', 'temperatura_c'])
fig_relacion.update_traces(marker=dict(size=8, opacity=0.8))
fig_relacion.show()

## 5. Conclusiones

* **En los datos:** El equipo opera de forma dinámica, oscilando en diferentes regímenes de velocidad (clústeres). La vibración RMS mantiene una correlación positiva directa con el aumento de las RPM y de la temperatura.
* **Qué registros considero anormales y por qué:** El modelo identificó 12 registros anormales que rompen la relación multivariable típica. Destacan dos tipos: 
    1. **Excesos absolutos:** Picos de vibración superiores a 4.5 mm/s en altas revoluciones.
    2. **Anomalías contextuales:** Registros donde la vibración superó los ~3.3 mm/s a pesar de operar a baja velocidad (~400 RPM). Esto es inusual e indica un probable problema mecánico de desbalanceo o desalineación.
* **Mejoras con más información:** 
    1. Si tuviera especificaciones del fabricante, aplicaría los límites de la norma ISO 10816 para clasificar la severidad.
    2. Si dispusiera de un histórico de fallas o bitácoras, cambiaría a un modelo supervisado (ej. *XGBoost*) para predecir modos de falla específicos.
    3. Si contara con mediciones de espectro de frecuencia, podría diagnosticar la causa raíz exacta de la falla (rodamientos, ejes, engranajes).